In [10]:
from langgraph.graph import StateGraph,START,END
from typing import TypedDict
from dotenv import load_dotenv
from langchain_groq import ChatGroq
from langgraph.checkpoint.memory import InMemorySaver

load_dotenv()

True

In [11]:
llm = ChatGroq(model="llama-3.3-70b-versatile",temperature=0.7)

In [12]:
class JokeState(TypedDict):
    topic:str
    joke:str
    explanation:str

In [13]:
def generate_joke(state : JokeState):

    prompt = f'Generate a joke on topic {state['topic']}'
    response = llm.invoke(prompt).content

    return {'topic':response}

In [14]:
def generate_explanation(state:JokeState):
    prompt = f'Generate an Explanation for the Joke - {state["joke"]}'
    response = llm.invoke(prompt).content
    return {'joke':response}

In [15]:
graph = StateGraph(JokeState)

In [16]:
graph.add_node('generate_joke',generate_joke)
graph.add_node('generate_explanation',generate_explanation)

In [17]:
graph.add_edge(START,'generate_joke')
graph.add_edge('generate_joke','generate_explanation')
graph.add_edge('generate_explanation',END)

In [18]:
checkpointer = InMemorySaver()

In [ ]:
workflow = graph.compile(checkpointer=checkpointer)

In [ ]:
config = {'configurable':{'thread_id':'thread1'}}
workflow.invoke({'topic':'Football'}, config=config)

In [ ]:
workflow.get_state(config)